load the notebook paths and standard library tools

In [ ]:
import hashlib
import json
import platform
import random
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "requirements.txt").is_file():
    project_root = project_root.parent

notebook_path = project_root / "notebooks" / "fibril_reproducibility.ipynb"
requirements_path = project_root / "requirements.txt"

record the seed and versions for an empty fibril run

In [ ]:
seed = 42

notebook = json.loads(notebook_path.read_text())
code_source = "\n\n".join(
    "".join(cell["source"])
    for cell in notebook["cells"]
    if cell["cell_type"] == "code"
)

run_metadata = {
    "stage": "fibril_environment",
    "code_version": hashlib.sha256(code_source.encode()).hexdigest(),
    "environment": {
        "python": platform.python_version(),
        "requirements_sha256": hashlib.sha256(requirements_path.read_bytes()).hexdigest(),
    },
    "data_version": None,
    "parameters": {"seed": seed},
    "seed_probe": random.Random(seed).getrandbits(64),
}
run_metadata["run_id"] = hashlib.sha256(
    json.dumps(run_metadata, sort_keys=True).encode()
).hexdigest()[:16]

save the run metadata for comparison across two executions

In [ ]:
results_path = project_root / "results" / "fibril_environment.json"
results_path.parent.mkdir(exist_ok=True)
results_path.write_text(json.dumps(run_metadata, indent=2, sort_keys=True) + "\n")
run_metadata